# Lab 1: Roofline and tiling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rht0111/edge-ai-arch-lab/blob/master/labs/lab1_roofline_tiling.ipynb)
&nbsp; Background: [course page for Lab 1](https://rht0111.github.io/edge-ai-arch-lab/labs/lab1/) &nbsp; Time: 25 min

Your edge chip has a 16x16 MAC array at 50 MHz (25.6 GOP/s peak) and a narrow
LPDDR link of 0.4 GB/s. Its on-chip buffer is **64 KiB**.

**Goals**

1. Place the operators of a ViT-Tiny block on the roofline of this chip.
2. Choose GEMM tile sizes that minimise DRAM traffic under the 64 KiB buffer.
3. Decide whether the attention GEMMs are memory-bound.

In [ ]:
# Setup: installs archsim in JupyterLite or Colab; does nothing on a local install.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["archsim", "ipywidgets"])
elif "google.colab" in sys.modules:
    %pip install -q "archsim @ git+https://github.com/rht0111/edge-ai-arch-lab"

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt

from archsim import energy, roofline, workloads
from archsim.checks import check

NPU = roofline.EDGE_NPU

In [ ]:
print(NPU.name, f"| peak {NPU.peak_ops / 1e9:.1f} GOP/s | ridge {NPU.ridge:.0f} ops/byte")

## 1. The best case

With an infinite buffer, each tensor crosses the DRAM link once and an
operator's intensity is `op.intensity()` from Lab 0. This is the best the
chip can ever do.

In [ ]:
block = workloads.vit_block(seq=197)
pts = [(op.name, op.intensity()) for op in block]
ax = roofline.plot(NPU, pts)
ax.set_title("ViT-Tiny block, minimum traffic (int8)");

Every GEMM sits right of the ridge: compute-bound, if the buffer were large
enough. It is not: QKV alone has 110 kB of weights.

## 2. Tiling

With a finite buffer we compute C in tiles. We use this loop nest
(output-stationary at the buffer level):

```
for m0 in range(0, M, Tm):          # rows of C
    for n0 in range(0, N, Tn):      # columns of C; the C tile stays on chip
        for k0 in range(0, K, Tk):
            load A[m0:m0+Tm, k0:k0+Tk]   unless it is already in the buffer
            load B[k0:k0+Tk, n0:n0+Tn]   unless it is already in the buffer
            C_tile += A_tile @ B_tile
        store C[m0:m0+Tm, n0:n0+Tn]
```

The buffer holds one A tile, one B tile (`bits`-bit elements) and one C
tile of 32-bit accumulators. A tile is reloaded whenever the loop asks for a
different tile than the one already held. Edge tiles may be smaller than
Tm x Tk; count the elements actually moved.

**Task 1.** Buffer footprint in bytes of one A, one B and one C tile.

In [ ]:
def footprint(Tm, Tk, Tn, bits=8):
    # TODO
    raise NotImplementedError("TODO")

In [ ]:
check("lab1.footprint", footprint)

**Task 2.** DRAM traffic in bytes of the loop nest above.

Hints. Let `nm, nn, nk` be the number of tiles along M, N, K.
How many times is all of A read? It depends on whether the A tile changes
between two consecutive iterations of the inner loops. Same question for B.
C is written once, in `bits`-bit precision.

In [ ]:
def tiled_traffic(M, K, N, Tm, Tk, Tn, bits=8):
    """DRAM bytes moved by the tiled GEMM above."""
    b = bits / 8
    nm, nn, nk = math.ceil(M / Tm), math.ceil(N / Tn), math.ceil(K / Tk)
    # TODO: count reads of A, reads of B, writes of C
    raise NotImplementedError("TODO")

In [ ]:
check("lab1.traffic", tiled_traffic)

**Task 3.** Search the tile sizes. Try every combination of the candidates
below and keep those whose footprint fits the buffer; return
`(minimum traffic, (Tm, Tk, Tn))`. Ties can be broken any way you like.
Tm and Tn start at 16: a smaller output tile would leave most of the 16x16
array idle. Tk may be anything; in this loop nest it only matters through
whether all of K fits at once.

In [ ]:
def candidates(dim, smallest=1):
    """Powers of two from `smallest` up to (not including) dim, plus dim itself."""
    c, p = [], smallest
    while p < dim:
        c.append(p)
        p *= 2
    return c + [dim]


def best_tiling(M, K, N, buffer_bytes=64 * 1024, bits=8):
    best = (math.inf, None)
    # TODO: loop over candidates(M, 16), candidates(K), candidates(N, 16)
    raise NotImplementedError("TODO")

In [ ]:
check("lab1.best", best_tiling)

## 3. Back on the roofline

Now place every GEMM of the block with its *real* intensity under a 64 KiB
buffer. For attention we tile one head (the `count` heads are independent).

In [ ]:
print(f"{'op':<8}{'tile (Tm,Tk,Tn)':<18}{'traffic/min':>12}{'ops/B':>8}  bound")
real = []
for op in block:
    if not op.is_gemm:
        continue
    M, K, N = op.shape
    t, tile = best_tiling(M, K, N)
    minimum = op.bytes() / op.count
    inten = 2 * M * K * N / t
    real.append((op.name, inten))
    print(f"{op.name:<8}{str(tile):<18}{t / minimum:>12.2f}{inten:>8.1f}  {NPU.bound(inten)}")
ax = roofline.plot(NPU, real)
ax.set_title("GEMMs with a 64 KiB buffer (int8)");

Most GEMMs reach the minimum traffic with 64 KiB, at least at 197 tokens. Now
shrink the buffer and watch the DRAM energy of the whole block (all GEMMs),
compared with the energy of its MACs:

In [ ]:
mac_pj = sum(op.macs for op in block) * energy.mac_energy(8)
kib = [8, 16, 32, 64, 128, 256]
for seq, style in ((197, "o-"), (1024, "s--")):
    ops = [op for op in workloads.vit_block(seq=seq) if op.is_gemm]
    macs = sum(op.macs for op in ops) * energy.mac_energy(8)
    dram = [sum(best_tiling(*op.shape, buffer_bytes=k * 1024)[0] * op.count for op in ops)
            * energy.dram_energy(1) / macs for k in kib]
    plt.semilogx(kib, dram, style, base=2, label=f"seq={seq}")
plt.axhline(1, color="grey", ls=":")
plt.xlabel("buffer (KiB)"); plt.ylabel("DRAM energy / MAC energy"); plt.legend()
plt.title("GEMMs of one block, int8");

## 4. Is attention memory-bound?

Look at the `scores` GEMM (Q·Kᵀ of one head) for several sequence lengths
and buffer sizes. Write down your answer and the reason.

In [ ]:
for kib_ in (16, 64):
    for seq in (64, 197, 1024):
        t, tile = best_tiling(seq, 64, seq, buffer_bytes=kib_ * 1024)
        inten = 2 * seq * 64 * seq / t
        print(f"{kib_:>3} KiB  seq={seq:<5} intensity {inten:6.1f} ops/B -> {NPU.bound(inten)}-bound, "
              f"{100 * float(NPU.attainable(inten)) / NPU.peak_ops:5.1f}% of peak")

## Stretch

1. **Double buffering.** To overlap loads with compute you need two copies of
   the A and B tiles. Rerun Section 3 with the A/B part of the footprint
   doubled (or simply `buffer_bytes=32 * 1024`). Which GEMMs suffer most?
2. **Fusion.** If softmax is computed on chip right after Q·Kᵀ and P·V
   consumes it immediately, the seq x seq matrix never goes to DRAM
   (the idea behind FlashAttention, Dao et al. 2022). Estimate the traffic of
   the fused attention for seq=1024 and its intensity.
3. Change the loop order so that B (the weights) is the outer, stationary
   tile. Which GEMMs prefer it?

## Reflection

The lecture said "data reuse sets energy". Using the buffer-size plot and
your traffic/min column, explain in two sentences which property of a GEMM
shape decides how much a small buffer hurts, and what that means for the
SRAM area you would spend on a 1 mW chip.